# XGBoost Starter - LB 0.793
In this notebook we build and train an XGBoost model using @raddar Kaggle dataset from [here][1] with discussion [here][2]. Then we engineer features suggested by @huseyincot in his notebooks [here][3] and [here][4]. This XGB model achieves CV 0.792 LB 0.793! When training with XGB, we use a special XGB dataloader called `DeviceQuantileDMatrix` which uses a small GPU memory footprint. This allows us to engineer more additional columns and train with more rows of data. Our feature engineering is performed using [RAPIDS][5] on the GPU to create new features quickly.

[1]: https://www.kaggle.com/datasets/raddar/amex-data-integer-dtypes-parquet-format
[2]: https://www.kaggle.com/competitions/amex-default-prediction/discussion/328514
[3]: https://www.kaggle.com/code/huseyincot/amex-catboost-0-793
[4]: https://www.kaggle.com/code/huseyincot/amex-agg-data-how-it-created
[5]: https://rapids.ai/

# Load Libraries

In [2]:
# LOAD LIBRARIES
import pandas as pd, numpy as np # CPU libraries
import cupy, cudf # GPU libraries
import matplotlib.pyplot as plt, gc, os

print('RAPIDS version',cudf.__version__)

RAPIDS version 21.10.01


cudf / cupy : 각각 pandas와 numpy의 기능을 GPU에서 실핼할 수 있도록 만든 라이브러리  

gc, os : kaggle의 제한된 자원을 효율적으로 쓰기 위한 gc(garbage collector)

In [3]:
# VERSION NAME FOR SAVED MODEL FILES
VER = 1

# TRAIN RANDOM SEED
SEED = 42

# FILL NAN VALUE
NAN_VALUE = -127 # will fit in int8

# FOLDS PER MODEL
FOLDS = 5

# Process and Feature Engineer Train Data
We will load @raddar Kaggle dataset from [here][1] with discussion [here][2]. Then we will engineer features suggested by @huseyincot in his notebooks [here][3] and [here][4]. We will use [RAPIDS][5] and the GPU to create new features quickly.

[1]: https://www.kaggle.com/datasets/raddar/amex-data-integer-dtypes-parquet-format
[2]: https://www.kaggle.com/competitions/amex-default-prediction/discussion/328514
[3]: https://www.kaggle.com/code/huseyincot/amex-catboost-0-793
[4]: https://www.kaggle.com/code/huseyincot/amex-agg-data-how-it-created
[5]: https://rapids.ai/

In [4]:
def read_file(path = '', usecols = None):
    # LOAD DATAFRAME
    if usecols is not None: df = cudf.read_parquet(path, columns=usecols)
    else: df = cudf.read_parquet(path)
    # REDUCE DTYPE FOR CUSTOMER AND DATE
    df['customer_ID'] = df['customer_ID'].str[-16:].str.hex_to_int().astype('int64')
    df.S_2 = cudf.to_datetime( df.S_2 )
    # SORT BY CUSTOMER AND DATE (so agg('last') works correctly)
    #df = df.sort_values(['customer_ID','S_2'])
    #df = df.reset_index(drop=True)
    # FILL NAN
    df = df.fillna(NAN_VALUE) 
    print('shape of data:', df.shape)
    
    return df

print('Reading train data...')
TRAIN_PATH = '../input/amex-data-integer-dtypes-parquet-format/train.parquet'
train = read_file(path = TRAIN_PATH)

Reading train data...
shape of data: (5531451, 190)


usecols: 내가 지정한 특정 열만 골라서 읽어올 때 쓰는 옵션  

cudf.read_parquet : parquet은 columnar 저장 방식이라 압축률이 뛰어나도 CPU기반의 pandas보다 수십배 빠르게 GPU메모리로 직접 데이터를 올릴 수 있음  

customer_ID 변환: 긴 ID의 마지막 16자리만 가져와서 16진수를 정수(int64)로 바꿈  
S_2 날짜 변환: 시계열 데이터 타입으로 바꿈  

fillna를 -127로 채우는 이유 : 실제 데이터 범위에서는 잘 나오지 않는 가짜 결측치, XGBoost는 이 값을 '정보 없음'상태로 특별한 카테고리로 인식

In [ ]:
#train.head()

In [5]:
def process_and_feature_engineer(df):
    # FEATURE ENGINEERING FROM 
    # https://www.kaggle.com/code/huseyincot/amex-agg-data-how-it-created
    all_cols = [c for c in list(df.columns) if c not in ['customer_ID','S_2']]
    cat_features = ["B_30","B_38","D_114","D_116","D_117","D_120","D_126","D_63","D_64","D_66","D_68"]
    num_features = [col for col in all_cols if col not in cat_features]

    test_num_agg = df.groupby("customer_ID")[num_features].agg(['mean', 'std', 'min', 'max', 'last'])
    test_num_agg.columns = ['_'.join(x) for x in test_num_agg.columns]

    test_cat_agg = df.groupby("customer_ID")[cat_features].agg(['count', 'last', 'nunique'])
    test_cat_agg.columns = ['_'.join(x) for x in test_cat_agg.columns]

    df = cudf.concat([test_num_agg, test_cat_agg], axis=1)
    del test_num_agg, test_cat_agg
    print('shape after engineering', df.shape )
    
    return df

train = process_and_feature_engineer(train)

shape after engineering (458913, 918)


all_cols: 고객 id와 날짜만 제외(id와 날짜는 관계 없음)한 전체 컬럼  
CAT과 NUM features의 분류

기존 컬럼수: customer_ID, S_2, CAT(11), NUM(177)
num_features에 대해 groupby id에 대해 mean,std,min,max,last를 구함 (177 * 5 ) = 885  
cat_features에 대해 groupby id에 대해 count, lsat, nunique를 구함 (11 * 3 ) = 33

.agg를 하면 컬럼 이름이 ('P_2', 'mean') 처럼 이중구조, 이를 '_'.join으로 P_2_mean으로 바꿔줌  

* 그럼 기존 컬럼에 있던 정보는 다 날라가고 agg의 요약값만 남는데 괜찮나?  
(평균, 표준편차, 최솟값, 최댓값, 마지막 값) (기록 개수, 가장 마지막 기록, unique 개수)  



In [ ]:
# ADD TARGETS
targets = cudf.read_csv('../input/amex-default-prediction/train_labels.csv')
targets['customer_ID'] = targets['customer_ID'].str[-16:].str.hex_to_int().astype('int64')
targets = targets.set_index('customer_ID')
train = train.merge(targets, left_index=True, right_index=True, how='left')
train.target = train.target.astype('int8')
del targets

# NEEDED TO MAKE CV DETERMINISTIC (cudf merge above randomly shuffles rows)
train = train.sort_index().reset_index()

# FEATURES
FEATURES = train.columns[1:-1]
print(f'There are {len(FEATURES)} features!')

target값도 불러와서 붙여줌  
train = tarin.sort_index().reset_index()  -> cudf.merge는 결과를 내보낼 때 행의 순서를 무작위로 섞어버리는 경우가 있음, 하지만 교차 검증을 할때는 데이터의 순서가 일정해야 성능 측정이 가능하므로, 순서를 고정시키는 작업 추가  

customerID는 학습에 필요 없으니깐 뺌  

# Train XGB
We will train using `DeviceQuantileDMatrix`. This has a very small GPU memory footprint.

In [ ]:
# LOAD XGB LIBRARY
from sklearn.model_selection import KFold
import xgboost as xgb
print('XGB Version',xgb.__version__)

# XGB MODEL PARAMETERS
xgb_parms = { 
    'max_depth':4, 
    'learning_rate':0.05, 
    'subsample':0.8,
    'colsample_bytree':0.6, 
    'eval_metric':'logloss',
    'objective':'binary:logistic',
    'tree_method':'gpu_hist',
    'predictor':'gpu_predictor',
    'random_state':SEED
}

XGB는 정형 데이터에서 성능이 좋음  

파라미터:
max_depth: tree의 depth
subsample: 전체 데이터중 80%만으로 하나의 나무  
colsample_bytree: 전체 feature중 60%만으로 하나의 나무  
ojbective: 모델의 목적, 이진 분류 문제 (결과값은 0-1사이의 확률값)  
eval_metric: logloss, 로그 손실을 사용 정답에서 멀어질수록 큰 벌점  
Logloss = -1/N sum(i=1)^N (y_i/log(p_i) + (1 - y_i)/log(1 - p_i))


In [ ]:
# NEEDED WITH DeviceQuantileDMatrix BELOW
class IterLoadForDMatrix(xgb.core.DataIter):
    def __init__(self, df=None, features=None, target=None, batch_size=256*1024):
        self.features = features
        self.target = target
        self.df = df
        self.it = 0 # set iterator to 0 몇번째인지 체크
        self.batch_size = batch_size
        self.batches = int( np.ceil( len(df) / self.batch_size ) )
        super().__init__()

    def reset(self):
        '''Reset the iterator'''
        self.it = 0

    def next(self, input_data):
        '''Yield next batch of data.'''
        if self.it == self.batches:
            return 0 # Return 0 when there's no more batch.
        
        a = self.it * self.batch_size
        b = min( (self.it + 1) * self.batch_size, len(self.df) )
        dt = cudf.DataFrame(self.df.iloc[a:b])
        input_data(data=dt[self.features], label=dt[self.target]) #, weight=dt['weight'])
        self.it += 1
        return 1

XGB의 DataIter을 상속받아 사용, batch  
batch size는 256*1024 행씩 끊어서, gpu에 한번에 올라갈 수 있는 데이터만큼  
self.batches에 전체 데이터를 batch size로 나눠 몇번에 나눠야되는지  

a, b 로 시작점과 끝점 계산  
input_data는 xgb라이브러리에 있음, 

In [ ]:
# https://www.kaggle.com/kyakovlev
# https://www.kaggle.com/competitions/amex-default-prediction/discussion/327534
def amex_metric_mod(y_true, y_pred):

    labels     = np.transpose(np.array([y_true, y_pred]))
    labels     = labels[labels[:, 1].argsort()[::-1]]
    weights    = np.where(labels[:,0]==0, 20, 1)
    cut_vals   = labels[np.cumsum(weights) <= int(0.04 * np.sum(weights))]
    top_four   = np.sum(cut_vals[:,0]) / np.sum(labels[:,0])

    gini = [0,0]
    for i in [1,0]:
        labels         = np.transpose(np.array([y_true, y_pred]))
        labels         = labels[labels[:, i].argsort()[::-1]]
        weight         = np.where(labels[:,0]==0, 20, 1)
        weight_random  = np.cumsum(weight / np.sum(weight))
        total_pos      = np.sum(labels[:, 0] *  weight)
        cum_pos_found  = np.cumsum(labels[:, 0] * weight)
        lorentz        = cum_pos_found / total_pos
        gini[i]        = np.sum((lorentz - weight_random) * weight)

    return 0.5 * (gini[1]/gini[0] + top_four)

np.transpose(np.array([y_true, y_pred])) -> 실제 정답과 예측 확률을 옆으로 붙여 2열짜리 표를 만듬  
labels[:,1]를 기준으로 전체 데이터를 내림차순으로 정렬, 연체 확률이 높은 사람이 윗행으로  
wight를 실제 정답이 0(정상)이면 20점, 연체면 1점을 줌  

cut_val -> 가중치의 누적 합계(cumsum)가 전체 가중치 합의 상위 4%가 되는 지점까지만 데이터를 자름  
top_four -> 그 상위 4% 안에 들어있는 연체자 수 sum(cut_val[:,0]) 를 전체 연체자 수로 나눔,  
이렇게 하면 전체 연체자중 상위 4% 구역에서 잡힌 연체자의 비율  

for i in [1:0]:
지니 계수는 완벽한 정답 대비 내 모델이 얼마나 잘했는지를 비교, 그렇기에 i=1 일때, 모델의 에측값으로 정렬해서 모델의 실력 측정, i=0 일때, 실제 정답으로 정렬해서 완벽한 모델을 측정  

weight_random -> 아무 실력 없는 모델이 연체자를 찾아가는 기준선 (0에서 1까지 일직선으로 증가하는 모양)  
wight/np.sum(weight): 각 한명 한명이 전체 인구(가중치 합)에서 차지하는 비중 ( 정상은 20/전체, 연체자는 1/전체)  
이걸 위에서부터 차례대로 더함. 리스트의 맨 처음은 0에서 마지막 사람까지 다 확인하면 1.0이 됨  
가중치 기준으로 내가 몇%나 확인했는지만 확인하는 0~1사이의 숫자  

total_pos     -> 전체 연체자의 가중치 합  
lorentz       -> 모델이 세운 순서대로 내려가면서 연체자를 얼마나 빨리 발견했는지 나타내는 비율 (0~1)  
실력이 좋을수록 lorentz 곡선은 왼쪽 위로 가파르게 치솟음  
gini[i]       -> lorentz(내 곡선)과 weight_random(랜덤 직선)사이의 거리 차이를 다 더함  
이 면적이 넓을수록 랜덤 모델보다 훨씬 빠르게 연체자를 찾아냈다는 뜻이고, 지니 점수는 올라감  

return        -> gini[1]/gini[0] 정규화된 지니 계수 (완벽한 정답 대비 내 모델의 상대적 줄 세우기 실력)  
top_four      -> 상위 4% 검출 능력  



* 정규화된 지니 계수 (Normalized GINI)  
Gini=0: 모델이 연체자와 정상인을 아예 구분 못함, Gini=1: 모델이 모든 연체자를 100% 확률로 맞히고, 정산인을 0%로 완벽 분리  
시각적 이해: 면적  
대각선(직선): 아무런 변별력이 없는 '무작위 모델'의 기준선  
곡선: 우리 모델이 예측한 순서대로 사람들을 세웠을 때, 실제로 연체자가 얼마나 빨리 발견되는지 나타내는 곡선  
Gini 점수: 대각선과 모델 곡선 사이의 면적을 계산한 값, 면적이 넓을수록 지니 계수가 커짐  
Gini 계수 + 연체 상위 4% 검출력을 합쳐서 점수를 매김  

In [ ]:
# 각 fold에서 중요했던 feature들을 담을 리스트
importances = [] 
# out of fold 예측값
oof = []
# GPU 메모리에서 GPU메모리로 옮김 (전체 데이터를 GPU에 계속 띄워둔 상태에서 XGBoost 학습하면 메모리 부족)
# 큰 데이터는 CPU에 두고, 학습할때만 아까 만든 DataIter 이용
train = train.to_pandas() # free GPU memory
TRAIN_SUBSAMPLE = 1.0
# 필요없는 메모리 강제로 비움
gc.collect()

# 5 fold-validation
skf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
for fold,(train_idx, valid_idx) in enumerate(skf.split(
            train, train.target )):
    
    # TRAIN WITH SUBSAMPLE OF TRAIN FOLD DATA
    # 전체 데이터로 학습하지 않고 일부만 사용할 때 (파라미터를 바꿔보고 이것저것 실험할때 이용 가능)
    if TRAIN_SUBSAMPLE<1.0:
        np.random.seed(SEED)
        train_idx = np.random.choice(train_idx, 
                       int(len(train_idx)*TRAIN_SUBSAMPLE), replace=False)
        np.random.seed(None)
    
    print('#'*25)
    print('### Fold',fold+1)
    print('### Train size',len(train_idx),'Valid size',len(valid_idx))
    print(f'### Training with {int(TRAIN_SUBSAMPLE*100)}% fold data...')
    print('#'*25)
    
    # TRAIN, VALID, TEST FOR FOLD K
    # Xy_train으로 훈련데이터를 batch_size만큼 쪼개서 준비함
    Xy_train = IterLoadForDMatrix(train.loc[train_idx], FEATURES, 'target')
    X_valid = train.loc[valid_idx, FEATURES]
    y_valid = train.loc[valid_idx, 'target']
    
    dtrain = xgb.DeviceQuantileDMatrix(Xy_train, max_bin=256)
    dvalid = xgb.DMatrix(data=X_valid, label=y_valid)
    
    # TRAIN MODEL FOLD K
    model = xgb.train(xgb_parms, 
                dtrain=dtrain,
                evals=[(dtrain,'train'),(dvalid,'valid')],
                num_boost_round=9999,
                early_stopping_rounds=100,
                verbose_eval=100) 
    model.save_model(f'XGB_v{VER}_fold{fold}.xgb')
    
    # GET FEATURE IMPORTANCE FOR FOLD K
    dd = model.get_score(importance_type='weight')
    df = pd.DataFrame({'feature':dd.keys(),f'importance_{fold}':dd.values()})
    importances.append(df)
            
    # INFER OOF FOLD K
    oof_preds = model.predict(dvalid)
    acc = amex_metric_mod(y_valid.values, oof_preds)
    print('Kaggle Metric =',acc,'\n')
    
    # SAVE OOF
    df = train.loc[valid_idx, ['customer_ID','target'] ].copy()
    df['oof_pred'] = oof_preds
    oof.append( df )
    
    del dtrain, Xy_train, dd, df
    del X_valid, y_valid, dvalid, model
    _ = gc.collect()
    
print('#'*25)
oof = pd.concat(oof,axis=0,ignore_index=True).set_index('customer_ID')
acc = amex_metric_mod(oof.target.values, oof.oof_pred.values)
print('OVERALL CV Kaggle Metric =',acc)

In [ ]:
# CLEAN RAM
del train
_ = gc.collect()

# Save OOF Preds

In [ ]:
oof_xgb = pd.read_parquet(TRAIN_PATH, columns=['customer_ID']).drop_duplicates()
oof_xgb['customer_ID_hash'] = oof_xgb['customer_ID'].apply(lambda x: int(x[-16:],16) ).astype('int64')
oof_xgb = oof_xgb.set_index('customer_ID_hash')
oof_xgb = oof_xgb.merge(oof, left_index=True, right_index=True)
oof_xgb = oof_xgb.sort_index().reset_index(drop=True)
oof_xgb.to_csv(f'oof_xgb_v{VER}.csv',index=False)
oof_xgb.head()

In [ ]:
# PLOT OOF PREDICTIONS
plt.hist(oof_xgb.oof_pred.values, bins=100)
plt.title('OOF Predictions')
plt.show()

In [ ]:
# CLEAR VRAM, RAM FOR INFERENCE BELOW
del oof_xgb, oof
_ = gc.collect()

# Feature Importance

In [ ]:
import matplotlib.pyplot as plt

df = importances[0].copy()
for k in range(1,FOLDS): df = df.merge(importances[k], on='feature', how='left')
df['importance'] = df.iloc[:,1:].mean(axis=1)
df = df.sort_values('importance',ascending=False)
df.to_csv(f'xgb_feature_importance_v{VER}.csv',index=False)

In [ ]:
NUM_FEATURES = 20
plt.figure(figsize=(10,5*NUM_FEATURES//10))
plt.barh(np.arange(NUM_FEATURES,0,-1), df.importance.values[:NUM_FEATURES])
plt.yticks(np.arange(NUM_FEATURES,0,-1), df.feature.values[:NUM_FEATURES])
plt.title(f'XGB Feature Importance - Top {NUM_FEATURES}')
plt.show()

# Process and Feature Engineer Test Data
We will load @raddar Kaggle dataset from [here][1] with discussion [here][2]. Then we will engineer features suggested by @huseyincot in his notebooks [here][1] and [here][4]. We will use [RAPIDS][5] and the GPU to create new features quickly.

[1]: https://www.kaggle.com/datasets/raddar/amex-data-integer-dtypes-parquet-format
[2]: https://www.kaggle.com/competitions/amex-default-prediction/discussion/328514
[3]: https://www.kaggle.com/code/huseyincot/amex-catboost-0-793
[4]: https://www.kaggle.com/code/huseyincot/amex-agg-data-how-it-created
[5]: https://rapids.ai/

In [ ]:
# CALCULATE SIZE OF EACH SEPARATE TEST PART
def get_rows(customers, test, NUM_PARTS = 4, verbose = ''):
    chunk = len(customers)//NUM_PARTS
    if verbose != '':
        print(f'We will process {verbose} data as {NUM_PARTS} separate parts.')
        print(f'There will be {chunk} customers in each part (except the last part).')
        print('Below are number of rows in each part:')
    rows = []

    for k in range(NUM_PARTS):
        if k==NUM_PARTS-1: cc = customers[k*chunk:]
        else: cc = customers[k*chunk:(k+1)*chunk]
        s = test.loc[test.customer_ID.isin(cc)].shape[0]
        rows.append(s)
    if verbose != '': print( rows )
    return rows,chunk

# COMPUTE SIZE OF 4 PARTS FOR TEST DATA
NUM_PARTS = 4
TEST_PATH = '../input/amex-data-integer-dtypes-parquet-format/test.parquet'

print(f'Reading test data...')
test = read_file(path = TEST_PATH, usecols = ['customer_ID','S_2'])
customers = test[['customer_ID']].drop_duplicates().sort_index().values.flatten()
rows,num_cust = get_rows(customers, test[['customer_ID']], NUM_PARTS = NUM_PARTS, verbose = 'test')

# Infer Test

In [ ]:
# INFER TEST DATA IN PARTS
skip_rows = 0
skip_cust = 0
test_preds = []

for k in range(NUM_PARTS):
    
    # READ PART OF TEST DATA
    print(f'\nReading test data...')
    test = read_file(path = TEST_PATH)
    test = test.iloc[skip_rows:skip_rows+rows[k]]
    skip_rows += rows[k]
    print(f'=> Test part {k+1} has shape', test.shape )
    
    # PROCESS AND FEATURE ENGINEER PART OF TEST DATA
    test = process_and_feature_engineer(test)
    if k==NUM_PARTS-1: test = test.loc[customers[skip_cust:]]
    else: test = test.loc[customers[skip_cust:skip_cust+num_cust]]
    skip_cust += num_cust
    
    # TEST DATA FOR XGB
    X_test = test[FEATURES]
    dtest = xgb.DMatrix(data=X_test)
    test = test[['P_2_mean']] # reduce memory
    del X_test
    gc.collect()

    # INFER XGB MODELS ON TEST DATA
    model = xgb.Booster()
    model.load_model(f'XGB_v{VER}_fold0.xgb')
    preds = model.predict(dtest)
    for f in range(1,FOLDS):
        model.load_model(f'XGB_v{VER}_fold{f}.xgb')
        preds += model.predict(dtest)
    preds /= FOLDS
    test_preds.append(preds)

    # CLEAN MEMORY
    del dtest, model
    _ = gc.collect()

# Create Submission CSV

In [ ]:
# WRITE SUBMISSION FILE
test_preds = np.concatenate(test_preds)
test = cudf.DataFrame(index=customers,data={'prediction':test_preds})
sub = cudf.read_csv('../input/amex-default-prediction/sample_submission.csv')[['customer_ID']]
sub['customer_ID_hash'] = sub['customer_ID'].str[-16:].str.hex_to_int().astype('int64')
sub = sub.set_index('customer_ID_hash')
sub = sub.merge(test[['prediction']], left_index=True, right_index=True, how='left')
sub = sub.reset_index(drop=True)

# DISPLAY PREDICTIONS
sub.to_csv(f'submission_xgb_v{VER}.csv',index=False)
print('Submission file shape is', sub.shape )
sub.head()

In [ ]:
# PLOT PREDICTIONS
plt.hist(sub.to_pandas().prediction, bins=100)
plt.title('Test Predictions')
plt.show()